# A simple livability index for Buenos Aires

Dimensions:
- Accessibility and mobility
- Green spaces and recreation
- Public services and amenities
- Safety and security
- Economic opportunities

## Prototype: computing metrics for the entire city (no disaggregation)

In [0]:
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

CRS = "EPSG:9498"

### Comunas (districts)

In [0]:
comunas = []

for i in range(1, 16):
    comunas.append(ox.geocode_to_gdf(f"Comuna {i}, CABA"))

In [0]:
comunas[0].name = "Comuna 1"

comunas = pd.concat(comunas)

comunas["comuna_n"] = comunas["name"].str.extract(r"(\d+)").astype(int)

In [0]:
fig = comunas.plot(
    "comuna_n",
    legend=True,
    edgecolor="black",
    linewidth=0.5,
    categorical=True,
    legend_kwds={"labels": comunas.name.array},
)

fig.get_legend().set_bbox_to_anchor((1.5, 1))
fig.get_legend().set_frame_on(False)
fig.set_frame_on(False)

### Coverage of the different dimensions across the city

#### Accessibility and mobility

In [0]:
# For the roads we can assume almost full coverage (we have seen it in previous notebooks)
#   roads = ox.graph_from_place(place, network_type='all')
# Also, it is a bad metric for Buenos Aires because:
#   a) It is hard to get, since the graph is too big for the whole city. It will delay downloading the data too much.
#   b) Public transport is way more important in dense cities for accesibility and mobility. 

public_transport = ox.features_from_place("CABA", tags={"public_transport": True})

public_transport["public_transport"].value_counts().plot(kind="bar")
plt.show()

In [0]:
public_transport.plot("public_transport", markersize=1)
plt.show()

#### Green spaces and recreation

In [0]:
parks_and_recreation = ox.features_from_place("CABA", tags={"leisure": True})

parks_and_recreation["leisure"].value_counts().plot(kind="bar")
plt.show()

In [0]:
parks_and_recreation.plot("leisure", markersize=1)
plt.show()

#### Public services

In [0]:
healthcare = ox.features_from_place("CABA", tags={"amenity": ["hospital", "pharmacy"]})
education = ox.features_from_place("CABA", tags={"amenity": ["kindergarten", "school", "university"]})

pd.concat([healthcare, education]).plot("amenity", legend=True, markersize=1)
plt.show()

#### Safety and security

In [0]:
emergency_services = ox.features_from_place(
    "CABA", tags={"amenity": ["police", "fire_station", "clinic"]}
)
street_lighting = ox.features_from_place("CABA", tags={'highway': 'street_lamp'})

pd.concat([emergency_services, street_lighting]).plot(markersize=1)
plt.show() # It seems we have a bad coverage here, mainly in street_lamp which is not useful

#### Economic opportunities

In [0]:
commerce = ox.features_from_place("CABA", tags={"shop": True})
employment = ox.features_from_place("CABA", tags={"office": True, "industrial": True})

pd.concat([commerce, employment]).plot(markersize=1)
plt.show()

### Metrics for the entire City of Buenos Aires

In [0]:
city_gdf = ox.geocode_to_gdf("CABA")
city_area = (city_gdf.to_crs(CRS).geometry.area / 1000 ** 2)[0]

In [0]:
# Accesibility and mobility
public_transport_density = len(public_transport) / city_area

print("Public transport density:", public_transport_density, "\n")

# Green spaces and recreation
parks_and_recreation_density = len(parks_and_recreation) / city_area
parks_and_recreation_coverage = parks_and_recreation.to_crs(CRS).area.sum() / (city_area * 1000 ** 2)

print("Parks and recreation density:", parks_and_recreation_density)
print("Parks and recreation coverage:", parks_and_recreation_coverage, "\n")

# Public services
healthcare_density = len(healthcare) / city_area
education_density = len(education) / city_area

print("Healthcare density:", healthcare_density)
print("Education density:", education_density, "\n")

# Emergency services and infrastructure
emergency_services_density = len(emergency_services) / city_area

print("Emergency services density:", emergency_services_density, "\n")

# Commerce and employment
commerce_density = len(commerce) / city_area
employment_density = len(employment) / city_area

print("Commerce density:", commerce_density)
print("Employment density:", employment_density)